In [1]:
import torch
import torchaudio

print(torch.__version__)
print(torchaudio.__version__)

2.8.0+cu128
2.8.0+cu128


In [2]:
from fairseq2 import gang
gang._thread_local.current_gangs = []

In [3]:
from omnilingual_asr.models.inference.pipeline import ASRInferencePipeline
pipeline = ASRInferencePipeline(model_card= 'omniASR_LLM_300M')

Output()

In [6]:
import subprocess

def encode_to_wav(audio):
    encoded_audio = subprocess.run(
        ['ffmpeg', '-i', audio, '-f', 'wav', 'pipe:1'],
        check = True,
        capture_output= True
    )
    return encoded_audio.stdout #stdout is the actual file

audio = 'sw-test-speech-1.m4a'
encoded_audio = encode_to_wav(audio)
type(encoded_audio)

bytes

In [22]:
import io, soundfile as sf
#obtained bytes go to a memory like object

audio = io.BytesIO(encoded_audio)
audio.seek(0)

waveform, sr = sf.read(audio)
waveform = torch.from_numpy(waveform).float()

In [26]:
print(f'Sample rate: {sr}')

Sample rate: 48000


In [27]:
import tempfile

with tempfile.NamedTemporaryFile(suffix='.wav', delete= False) as tmp:
    sf.write(tmp.name, waveform.numpy(), sr)
    transcript = pipeline.transcribe([tmp.name], batch_size= 1)

print(transcript)

['mimi anaitwa nathan una akiukweli napenda walisamaki yaani akiongelea walisamaki naongelea ni rosten ni ule wale ambao yaani samaki wake unakuwa unaoroja uroja yaani unakuwa mtaa tunaopenda']


In [28]:
def segment_audio(waveform, sr, chunk_duration= 5, overlap= 0.5):
    """
    Args:
        waveform: audio data in torch.Tensor format
        sr: sample rate i.e, number of audio samples in a second
        chunk_duration: how long a chunk is, defaults to 5 as defined in this function
        overlap: overlap ratio (0-1), eg. o.5 overlap means 50% overlap

    Returns:
        List of tuples, containing the chunk data, it's start time and end time
    """
    chunk_size = int(sr * chunk_duration)
    hop_size = int(chunk_size * (1 - overlap)) #stride between chunks

    chunks = []
    start_idx = 0

    while start_idx < len(waveform):
        end_idx = min(start_idx + chunk_size, len(waveform)) #for the last chunk, it's usually not the full chunk size
        chunk = waveform[start_idx:end_idx]

        start_time = start_idx / sr
        end_time = end_idx / sr

        chunks.append((chunk, start_time, end_time))
        start_idx += hop_size

    return chunks

In [31]:
chunks = segment_audio(waveform, sr, chunk_duration= 3)
chunks[-1]

(tensor([[-8.1482e-03, -8.1482e-03],
         [-6.5613e-03, -6.5613e-03],
         [-4.9744e-03, -4.9744e-03],
         ...,
         [ 0.0000e+00,  0.0000e+00],
         [ 6.1035e-05,  6.1035e-05],
         [ 6.1035e-05,  6.1035e-05]]),
 16.5,
 17.536)

In [32]:
transcripts = []
for chunk, start_time, end_time in chunks:
    with tempfile.NamedTemporaryFile(suffix= '.wav', delete= False) as tmp:
        sf.write(tmp.name, chunk.numpy(), sr)
        transcript = pipeline.transcribe([tmp.name], batch_size= 1)
        transcripts.append({
            'text': transcript,
            'start': start_time,
            'end': end_time,
        })
        print(f"{start_time:.1f}s - {end_time:.1f}s: {transcript}")

0.0s - 3.0s: ['mimi anaitwa nathan anaakiukweli na']
1.5s - 4.5s: ['sanaki ukweli napenda wali sanaki']
3.0s - 6.0s: ['penda wali sanati yani']
4.5s - 7.5s: ['yeniye ki konkine wali samanthi']
6.0s - 9.0s: ['onkireer walisanafi nonkireer roo']
7.5s - 10.5s: ['no gre ali rosten']
9.0s - 12.0s: ['osteni ni ule wale ambao']
10.5s - 13.5s: ['ni ule wale ambao wenye samaki wakio na kwa uneno']
12.0s - 15.0s: ['ni samaki wake unakuwa unae unaroja uroja yenu unakuwa']
13.5s - 16.5s: ['وروج وروج ان کو ہفتہ م تھونا']
15.0s - 17.5s: ['tamu tunawupenda']
16.5s - 17.5s: ['ópɛndɔ']
